In [95]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import joblib
import nltk
from sklearn.metrics import accuracy_score,classification_report, confusion_matrix
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

In [96]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to C:\Users\Mann
[nltk_data]     Raval\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [97]:
df = pd.read_csv("data/WELFake_Dataset.csv")

In [98]:
print("Initial shape:", df.shape)
df.head()

Initial shape: (72134, 4)


,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


# EDA

In [99]:
print("\nLabel Distribution:")
print(df['label'].value_counts())

print("\nMissing Values:")
print(df.isnull().sum())


Label Distribution:
label
1    37106
0    35028
Name: count, dtype: int64

Missing Values:
Unnamed: 0      0
title         558
text           39
label           0
dtype: int64


# Text Cleaning

In [100]:
def clean_text(text):

    text = str(text).lower()

    text = re.sub(r'[^a-zA-Z]', ' ', text)

    text = re.sub(r'\s+', ' ', text)

    return text

In [101]:
df['content'] = df['title'].fillna('') + " " + df['text'].fillna('')
df['content'] = df['content'].apply(clean_text)
df = df[['content', 'label']]
df.dropna(inplace=True)
print("\nAfter Cleaning:", df.shape)


After Cleaning: (72134, 2)


In [102]:
df.head()

,content,label
0,law enforcement on high alert following threat...,1
1,did they post their votes for hillary already,1
2,unbelievable obama s attorney general says mos...,1
3,bobby jindal raised hindu uses story of christ...,0
4,satan russia unvelis an image of its terrifyin...,1


# Train-Test Split

In [103]:
X = df['content']
y = df['label']

In [104]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

# TF-IDF Vectorization

In [105]:
print("\nVectorizing text...")
vectorizer = TfidfVectorizer(
    stop_words='english',
    max_df=0.8,
    min_df=5,
    ngram_range=(1,2),
    max_features=60000
)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("Vector shape:", X_train_vec.shape)


Vectorizing text...
Vector shape: (57707, 60000)


# Train Multiple Models

In [106]:
models = {

    "Logistic Regression": LogisticRegression(max_iter=1000),

    "Naive Bayes": MultinomialNB(),

    "SVM": LinearSVC()
}

In [107]:
results = {}
print("\nTraining Models...\n")

for name, model in models.items():

    print("Training:", name)

    model.fit(X_train_vec, y_train)

    preds = model.predict(X_test_vec)

    acc = accuracy_score(y_test, preds)

    results[name] = acc

    print("Accuracy:", acc)

    print(classification_report(y_test, preds))


Training Models...

Training: Logistic Regression
Accuracy: 0.9524502668607472
              precision    recall  f1-score   support

           0       0.96      0.94      0.95      7006
           1       0.95      0.96      0.95      7421

    accuracy                           0.95     14427
   macro avg       0.95      0.95      0.95     14427
weighted avg       0.95      0.95      0.95     14427

Training: Naive Bayes
Accuracy: 0.8768281694045886
              precision    recall  f1-score   support

           0       0.88      0.86      0.87      7006
           1       0.87      0.89      0.88      7421

    accuracy                           0.88     14427
   macro avg       0.88      0.88      0.88     14427
weighted avg       0.88      0.88      0.88     14427

Training: SVM
Accuracy: 0.9688084840923269
              precision    recall  f1-score   support

           0       0.97      0.96      0.97      7006
           1       0.96      0.98      0.97      7421

    accu

In [108]:
best_model_name = max(results, key=results.get)
best_accuracy = results[best_model_name]
print("\nBest Model:", best_model_name)
print("Best Accuracy:", best_accuracy)
best_model = models[best_model_name]


Best Model: SVM
Best Accuracy: 0.9688084840923269


# Hyperparameter Tuning

In [109]:
print("\nHyperparameter Tuning...")

if best_model_name == "Logistic Regression":

    params = {'C': [0.1, 1, 10]}

    grid = GridSearchCV(
        LogisticRegression(max_iter=1000),
        params,
        cv=5,
        n_jobs=-1
    )

elif best_model_name == "SVM":

    params = {'C': [0.1, 1, 10]}

    grid = GridSearchCV(
        LinearSVC(),
        params,
        cv=5,
        n_jobs=-1
    )

else:  # Naive Bayes

    params = {'alpha': [0.1, 0.5, 1.0]}

    grid = GridSearchCV(
        MultinomialNB(),
        params,
        cv=5
    )


grid.fit(X_train_vec, y_train)

best_model = grid.best_estimator_

print("Best Params:", grid.best_params_)


Hyperparameter Tuning...
Best Params: {'C': 1}


# Final Evaluation

In [113]:
final_preds = best_model.predict(X_test_vec)

print("\nFinal Model Evaluation:\n")

print("Accuracy:", accuracy_score(y_test, final_preds))

print("Confusion Matrix:")
print(confusion_matrix(y_test, final_preds))

print("\nClassification Report:")
print(classification_report(y_test, final_preds))


Final Model Evaluation:

Accuracy: 0.9688084840923269
Confusion Matrix:
[[6734  272]
 [ 178 7243]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.96      0.97      7006
           1       0.96      0.98      0.97      7421

    accuracy                           0.97     14427
   macro avg       0.97      0.97      0.97     14427
weighted avg       0.97      0.97      0.97     14427



# Save Model

In [114]:
joblib.dump(best_model, "model/fake_news_model.pkl")
joblib.dump(vectorizer, "model/tfidf.pkl")

print("\nModel saved successfully in model/ folder")



Model saved successfully in model/ folder


# What??

In [115]:
if best_model_name == "SVM":

    print("\nTop Important Words (SVM):")

    feature_names = vectorizer.get_feature_names_out()
    coefs = best_model.coef_[0]

    top_fake = np.argsort(coefs)[:15]
    top_real = np.argsort(coefs)[-15:]

    print("\nWords indicating FAKE:")
    for i in top_fake:
        print(feature_names[i])

    print("\nWords indicating REAL:")
    for i in top_real:
        print(feature_names[i])



Top Important Words (SVM):

Words indicating FAKE:
reuters
breitbart
washington reuters
said
president donald
follow
york times
twitter
follow twitter
thursday
reuters president
monday
tuesday
factbox
friday

Words indicating REAL:
wire
november
hillary
read
com
entire story
getty images
video
breaking
getty
october
image
twitter com
featured
featured image
